In [ ]:
# JERICHON import bootstrap (shared resolver in src/project_paths.py)
from pathlib import Path
import os, sys

# Locate repository from the current working directory or its parents.
_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
for _base in list(_candidates):
    try:
        _candidates.extend(_child for _child in _base.iterdir() if _child.is_dir())
    except (OSError, PermissionError):
        pass
PROJECT_ROOT = next((p.resolve() for p in _candidates
                     if (p / "src" / "__init__.py").is_file()
                     and (p / "src" / "data" / "builders").is_dir()), None)
if PROJECT_ROOT is None:
    _env_root = os.environ.get("JERICHON_PROJECT_ROOT")
    if _env_root:
        _candidate = Path(_env_root).expanduser().resolve()
        if (_candidate / "src" / "__init__.py").is_file():
            PROJECT_ROOT = _candidate
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не найден корень JERICHON. Откройте VS Code в папке репозитория "
        "или задайте JERICHON_PROJECT_ROOT.")
if str(PROJECT_ROOT) in sys.path:
    sys.path.remove(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))
from src.project_paths import bootstrap_project, find_project_root
PROJECT_ROOT = bootstrap_project(PROJECT_ROOT)


# JERICHON — Trade Sequence & Order-Flow Microstructure (короткое исследование)

**Вопрос:** содержит ли последовательность сделок и дисбаланс заявок информацию о направлении ближайшего изменения цены?

На картинке показана интуиция queue depletion/replenishment: market-sell объём `Mˢ` уменьшает bid, limit-buy объём `Lᵇ` увеличивает bid; грубая оценка изменения bid выражается через `(Lᵇ − Mˢ)/D`. Это полезная иллюстрация механики очереди, но не универсальная формула OFI и не готовая модель mid-price.

План ноутбука:
1. Прочитать **сырые** `trades` и `orderbook` из локального AlgoPack.
2. Зафиксировать фактическую схему полей — ничего не угадывать молча.
3. Построить signed trade flow / imbalance из последовательности сделок.
4. Если исторический orderbook содержит bid/ask snapshots с размерами — рассчитать best-level OFI и проверить связь с ближайшими изменениями mid-price.
5. Оценить горизонт в **следующих изменениях mid-price**, а не только в минутах/барах.

`tradestats/obstats/orderstats` в этом эксперименте не используются: цель — проверить сырой поток, а не ещё раз агрегированные Super Candles.

## 0. Параметры и ограничения
Начинаем с одного тикера и небольшого периода. Увеличивать выборку будем только после проверки схемы и корректности timestamp.

In [1]:

from pathlib import Path
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

TICKER = "GAZP"
PROJECT_ROOT = None
MAX_DAYS = None                 # например 5 для короткого пилота; None = все локальные даты
MAX_TRADES = 500_000            # ограничение RAM; можно увеличить после проверки
HORIZONS_NEXT_QUOTES = (1, 2, 3, 5, 10)
# Явные переопределения после просмотра schema_report. Оставить None для автоопределения.
TRADE_TS_COL = None
TRADE_PRICE_COL = None
TRADE_SIZE_COL = None
TRADE_SIDE_COL = None
BOOK_TS_COL = None
BID_PRICE_COL = None
ASK_PRICE_COL = None
BID_SIZE_COL = None
ASK_SIZE_COL = None


## 1. Найти локальную историю и прочитать только raw trades / orderbook

In [2]:

def find_project_root(start=None):
    if PROJECT_ROOT is not None:
        p = Path(PROJECT_ROOT).expanduser().resolve()
        if not p.exists(): raise FileNotFoundError(p)
        return p
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p/"data"/"for_ml"/"storage"/"history").exists():
            return p
    legacy = Path(r"C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON")
    if (legacy/"data"/"for_ml"/"storage"/"history").exists(): return legacy
    raise FileNotFoundError("Не найден JERICHON. Открой ноутбук из репозитория или задай PROJECT_ROOT.")

ROOT = find_project_root()
DATA_ROOT = ROOT/"data" if (ROOT/"data").exists() else ROOT
HISTORY = DATA_ROOT/"for_ml"/"storage"/"history"
trade_files = sorted((HISTORY/"realtime"/TICKER).glob("*/trades.parquet"))
book_files = sorted((HISTORY/"realtime"/TICKER).glob("*/orderbook.parquet"))
if MAX_DAYS:
    trade_files, book_files = trade_files[-MAX_DAYS:], book_files[-MAX_DAYS:]
if not trade_files:
    raise FileNotFoundError(f"Нет trades.parquet в {HISTORY/'realtime'/TICKER}")
def read_many(paths, limit=None):
    frames=[]
    for p in paths:
        x=pd.read_parquet(p)
        if not x.empty:
            x=x.copy(); x["__file_date"]=p.parent.name
            frames.append(x)
    if not frames: return pd.DataFrame()
    out=pd.concat(frames,ignore_index=True)
    if limit and len(out)>limit: out=out.tail(limit).copy()
    return out
trades=read_many(trade_files,MAX_TRADES)
orderbook=read_many(book_files,None)
print("trades:",trades.shape,"orderbook:",orderbook.shape)


trades: (500000, 14) orderbook: (460, 8)


## 2. Схема источников — обязательная проверка до расчётов
Ноутбук покажет реальные названия, типы и примеры. Если автоопределение выбрало не те поля, укажи их в параметрах ячейки 0 и перезапусти с этого места.

In [3]:

def schema_report(df, source):
    rows=[]
    for c in df.columns:
        s=df[c]
        examples=s.dropna().astype(str).head(3).tolist()
        rows.append({"source":source,"column":str(c),"dtype":str(s.dtype),
                     "non_null":int(s.notna().sum()),"unique":int(s.nunique(dropna=True)),
                     "examples":" | ".join(examples)})
    return pd.DataFrame(rows)
schema=pd.concat([schema_report(trades,"trades"),schema_report(orderbook,"orderbook")],ignore_index=True)
display(schema)
print("Trade columns:",trades.columns.tolist())
print("Orderbook columns:",orderbook.columns.tolist())


,source,column,dtype,non_null,unique,examples
0,trades,TRADENO,int64,500000,500000,17915799449 | 17915799485 | 17915799486
1,trades,TRADETIME,object,500000,51858,19:01:46 | 19:01:48 | 19:01:48
2,trades,PRICE,float64,500000,655,113.79 | 113.81 | 113.81
3,trades,QUANTITY,int64,500000,2515,39 | 108 | 12
4,trades,PERIOD,object,500000,4,N | N | N
5,trades,VALUE,float64,500000,91479,44378.1 | 122914.8 | 13657.2
6,trades,BUYSELL,object,500000,2,S | B | B
7,trades,DECIMALS,int64,500000,1,2 | 2 | 2
8,trades,TRADINGSESSION,int64,500000,3,2 | 2 | 2
9,trades,TRADEDATE,object,500000,7,2026-10-23 | 2026-10-23 | 2026-10-23


Trade columns: ['TRADENO', 'TRADETIME', 'PRICE', 'QUANTITY', 'PERIOD', 'VALUE', 'BUYSELL', 'DECIMALS', 'TRADINGSESSION', 'TRADEDATE', 'TRADE_SESSION_DATE', 'ts', 'ticker', '__file_date']
Orderbook columns: ['BUYSELL', 'PRICE', 'QUANTITY', 'SEQNUM', 'UPDATETIME', 'DECIMALS', 'ticker', '__file_date']


## 3. Нормализация сделок и знак агрессора

Если в raw trades есть явный side/aggressor flag — используем его после проверки значений. Если его нет, используем tick rule как **приближение**, а не выдаём за реальный aggressor side: цена выше предыдущей сделки → buy-initiated, ниже → sell-initiated; при равенстве переносим последний известный знак.

In [4]:

def pick_col(df, explicit, aliases, required=True):
    if explicit:
        if explicit not in df.columns: raise KeyError(f"Нет колонки {explicit}")
        return explicit
    low={str(c).lower():c for c in df.columns}
    for a in aliases:
        if a.lower() in low: return low[a.lower()]
    if required: raise KeyError(f"Не найдено поле из {aliases}; фактические колонки: {list(df.columns)}")
    return None

t_ts=pick_col(trades,TRADE_TS_COL,["ts","timestamp","time","tradetime","datetime"])
t_px=pick_col(trades,TRADE_PRICE_COL,["price","prc","trade_price"])
t_sz=pick_col(trades,TRADE_SIZE_COL,["quantity","qty","volume","size"],False)
t_value=pick_col(trades,None,["value","turnover","trade_value"],False)
t_side=pick_col(trades,TRADE_SIDE_COL,["side","buysell","buy_sell","aggressor","direction","trade_side"],False)
if t_sz is None and t_value is None:
    raise KeyError("Не найдены ни quantity/qty/volume/size, ни VALUE/turnover. Сверь schema_report.")
tr=trades.copy()
tr["ts"]=pd.to_datetime(tr[t_ts],errors="coerce",utc=True)
tr["price"]=pd.to_numeric(tr[t_px],errors="coerce")
if t_sz is not None:
    tr["size"]=pd.to_numeric(tr[t_sz],errors="coerce")
    size_source=t_sz
else:
    # AlgoPack VALUE — денежный оборот сделки; quantity в единицах инструмента = VALUE / PRICE.
    tr["size"]=pd.to_numeric(tr[t_value],errors="coerce")/tr["price"]
    size_source=f"{t_value}/{t_px}"
print("Trade size source:",size_source)
tr=tr.dropna(subset=["ts","price","size"]).query("price > 0 and size > 0").sort_values("ts").reset_index(drop=True)
if t_side:
    raw=tr[t_side].astype(str).str.strip().str.lower()
    buy_tokens={"b","buy","ซื้อ","1","2","покупка","купля","buyer","bid"}
    sell_tokens={"s","sell","-1","3","продажа","продавец","seller","ask"}
    tr["sign"]=np.where(raw.isin(buy_tokens),1,np.where(raw.isin(sell_tokens),-1,np.nan))
    print("Explicit side field:",t_side,"| unmapped:",int(tr["sign"].isna().sum()))
else:
    tr["sign"]=np.nan
if tr["sign"].isna().mean()>0.05:
    tick=np.sign(tr["price"].diff())
    tick=tick.replace(0,np.nan).ffill()
    tr["sign"]=tr["sign"].fillna(tick).fillna(0)
    print("Tick rule used for missing/unknown aggressor labels; not ground-truth side.")
tr["signed_size"]=tr["sign"]*tr["size"]
tr["signed_notional"]=tr["signed_size"]*tr["price"]
print("Normalized trades:",len(tr),"| date range:",tr.ts.min(),tr.ts.max())
display(tr.head())


Trade size source: QUANTITY
Explicit side field: BUYSELL | unmapped: 0
Normalized trades: 500000 | date range: 2026-10-23 16:01:46+00:00 2026-11-03 16:32:08+00:00


,TRADENO,TRADETIME,PRICE,QUANTITY,PERIOD,VALUE,BUYSELL,DECIMALS,TRADINGSESSION,TRADEDATE,TRADE_SESSION_DATE,ts,ticker,__file_date,price,size,sign,signed_size,signed_notional
0,17915799449,19:01:46,113.79,39,N,44378.1,S,2,2,2026-10-23,2026-10-23,2026-10-23 16:01:46+00:00,GAZP,2026-10-23,113.79,39,-1.0,-39.0,-4437.81
1,17915799485,19:01:48,113.81,108,N,122914.8,B,2,2,2026-10-23,2026-10-23,2026-10-23 16:01:48+00:00,GAZP,2026-10-23,113.81,108,1.0,108.0,12291.48
2,17915799486,19:01:48,113.81,12,N,13657.2,B,2,2,2026-10-23,2026-10-23,2026-10-23 16:01:48+00:00,GAZP,2026-10-23,113.81,12,1.0,12.0,1365.72
3,17915799501,19:01:49,113.81,3,N,3414.3,B,2,2,2026-10-23,2026-10-23,2026-10-23 16:01:49+00:00,GAZP,2026-10-23,113.81,3,1.0,3.0,341.43
4,17915799534,19:01:51,113.79,26,N,29585.4,S,2,2,2026-10-23,2026-10-23,2026-10-23 16:01:51+00:00,GAZP,2026-10-23,113.79,26,-1.0,-26.0,-2958.54


## 4. Trade flow: последовательность, интенсивность и imbalance
Сначала используем окна по количеству сделок, чтобы не смешивать разную активность рынка с фиксированными минутами. Измеряем накопленный signed volume / total volume и signed notional / total notional.

In [5]:

# Кумулятивные rolling-признаки; значения в момент i используют только сделки <= i.
for w in (20,50,100):
    sv=tr["signed_size"].rolling(w,min_periods=max(5,w//4)).sum()
    tv=tr["size"].rolling(w,min_periods=max(5,w//4)).sum()
    sn=tr["signed_notional"].rolling(w,min_periods=max(5,w//4)).sum()
    tn=(tr["size"]*tr["price"]).rolling(w,min_periods=max(5,w//4)).sum()
    tr[f"trade_imbalance_{w}"]=sv/tv.replace(0,np.nan)
    tr[f"notional_imbalance_{w}"]=sn/tn.replace(0,np.nan)
    tr[f"trade_intensity_{w}"]=tr["ts"].diff().dt.total_seconds().rolling(w,min_periods=5).median()
display(tr[[f"trade_imbalance_{w}" for w in (20,50,100)]].describe().T)


,count,mean,std,min,25%,50%,75%,max
trade_imbalance_20,499996.0,-0.065974,0.752635,-1.0,-0.854545,-0.129318,0.715035,1.0
trade_imbalance_50,499989.0,-0.064211,0.663368,-1.0,-0.697897,-0.095143,0.548480,1.0
trade_imbalance_100,499976.0,-0.064749,0.599685,-1.0,-0.601566,-0.083332,0.455806,1.0


## 5. Orderbook: проверить snapshots и mid-price

Автоопределение работает только для очевидных полей. Если структура вложенная/level-based — остановись, изучи schema и явно задай mapping. Мы не будем делать вид, что bid/ask восстановлены, если в parquet другой формат.

In [6]:

book_ready=False
if not orderbook.empty:
    b_ts=pick_col(orderbook,BOOK_TS_COL,["ts","timestamp","time","datetime"],False)
    b_bid=pick_col(orderbook,BID_PRICE_COL,["bid","bid_price","best_bid","bidprice"],False)
    b_ask=pick_col(orderbook,ASK_PRICE_COL,["ask","ask_price","best_ask","askprice"],False)
    b_bsz=pick_col(orderbook,BID_SIZE_COL,["bid_size","bid_volume","bidsize","best_bid_size"],False)
    b_asz=pick_col(orderbook,ASK_SIZE_COL,["ask_size","ask_volume","asksize","best_ask_size"],False)
    print("Detected:",{"ts":b_ts,"bid":b_bid,"ask":b_ask,"bid_size":b_bsz,"ask_size":b_asz})
    if all([b_ts,b_bid,b_ask,b_bsz,b_asz]):
        ob=orderbook.copy()
        ob["ts"]=pd.to_datetime(ob[b_ts],errors="coerce",utc=True)
        for dst,src in [("bid",b_bid),("ask",b_ask),("bid_size",b_bsz),("ask_size",b_asz)]:
            ob[dst]=pd.to_numeric(ob[src],errors="coerce")
        ob=ob.dropna(subset=["ts","bid","ask","bid_size","ask_size"])
        ob=ob[(ob.bid>0)&(ob.ask>=ob.bid)&(ob.bid_size>=0)&(ob.ask_size>=0)].sort_values("ts")
        ob=ob.drop_duplicates("ts",keep="last").reset_index(drop=True)
        ob["mid"]=(ob.bid+ob.ask)/2
        ob["spread"]=ob.ask-ob.bid
        ob["depth_imbalance"]=(ob.bid_size-ob.ask_size)/(ob.bid_size+ob.ask_size).replace(0,np.nan)
        # Best-quote OFI, Cont et al. event contribution.
        bp,ap,bq,aq=ob.bid,ob.ask,ob.bid_size,ob.ask_size
        ofi=( (bp>=bp.shift(1))*bq - (bp<=bp.shift(1))*bq.shift(1)
             -(ap<=ap.shift(1))*aq + (ap>=ap.shift(1))*aq.shift(1) )
        ob["ofi_best"]=ofi
        ob["ofi_norm"]=ob["ofi_best"]/(ob["bid_size"]+ob["ask_size"]).replace(0,np.nan)
        book_ready=True
        print("Valid orderbook snapshots:",len(ob),"|",ob.ts.min(),ob.ts.max())
        display(ob[["ts","bid","ask","bid_size","ask_size","mid","spread","depth_imbalance","ofi_best"]].head())
    else:
        print("Не удалось однозначно сопоставить best bid/ask + sizes. OFI пропускаем; сначала изучи schema_report.")
else:
    print("Исторический orderbook отсутствует: mid-price/OFI из стакана посчитать нельзя.")


Detected: {'ts': None, 'bid': None, 'ask': None, 'bid_size': None, 'ask_size': None}
Не удалось однозначно сопоставить best bid/ask + sizes. OFI пропускаем; сначала изучи schema_report.


## 6. Проверка направления ближайших изменений цены

Если стакан восстановлен, используем mid-price и измеряем следующие изменения **наблюдаемого mid-price**. Если стакан не восстановлен, делаем только вспомогательный анализ будущего изменения цены сделки — это не mid-price и не эквивалент.

Сначала объединяем признаки сделок с ближайшим предшествующим snapshot (`merge_asof`, backward). Никаких будущих snapshots в признаках.

In [7]:

if book_ready:
    ob_features=ob[["ts","mid","spread","depth_imbalance","ofi_best","ofi_norm"]].copy()
    # Feature at trade time may use only the last snapshot at or before the trade.
    joined=pd.merge_asof(tr.sort_values("ts"),ob_features.sort_values("ts"),on="ts",direction="backward")
    price_series=ob["mid"].to_numpy()
    # Forward labels are next observed mid changes, not fixed clock bars.
    mid_index=ob[["ts","mid"]].copy()
    # Associate each trade with latest snapshot position.
    snap_pos=np.searchsorted(ob["ts"].astype("int64").to_numpy(),joined["ts"].astype("int64").to_numpy(),side="right")-1
    for h in HORIZONS_NEXT_QUOTES:
        fut=np.full(len(joined),np.nan)
        ok=(snap_pos>=0)&(snap_pos+h<len(ob))
        fut[ok]=price_series[snap_pos[ok]+h]/price_series[snap_pos[ok]]-1
        joined[f"future_mid_return_{h}"]=fut
    joined["target_mid_up_1"]=np.where(joined["future_mid_return_1"].notna(),
                                        (joined["future_mid_return_1"]>0).astype(int),np.nan)
    analysis_cols=["trade_imbalance_20","trade_imbalance_50","notional_imbalance_20",
                   "depth_imbalance","ofi_norm","spread"]
    available=[c for c in analysis_cols if c in joined]
    display(joined[available+[f"future_mid_return_{h}" for h in HORIZONS_NEXT_QUOTES]].describe().T)
else:
    joined=tr.copy()
    for h in HORIZONS_NEXT_QUOTES:
        joined[f"future_trade_return_{h}"]=joined["price"].shift(-h)/joined["price"]-1
    print("Внимание: ниже trade-price proxy, не mid-price.")


Внимание: ниже trade-price proxy, не mid-price.


## 7. Простая проверка предсказательной связи
Это диагностическая группировка по квантилям imbalance/OFI, а не ML и не подбор порога. Проверяем монотонность: растёт ли будущая mid-price доходность при усилении положительного дисбаланса?

In [8]:

if book_ready:
    target="future_mid_return_1"
    test_features=[c for c in ["trade_imbalance_20","notional_imbalance_20","depth_imbalance","ofi_norm"] if c in joined]
else:
    target="future_trade_return_1"
    test_features=["trade_imbalance_20","notional_imbalance_20"]
rows=[]
for feat in test_features:
    z=joined[[feat,target]].replace([np.inf,-np.inf],np.nan).dropna()
    if len(z)<1000: continue
    try: z["bucket"]=pd.qcut(z[feat],5,duplicates="drop")
    except Exception: continue
    g=z.groupby("bucket",observed=True)[target].agg(["count","mean","median"])
    g["feature"]=feat
    rows.append(g.reset_index())
if rows:
    bucket_results=pd.concat(rows,ignore_index=True)
    display(bucket_results)
else:
    bucket_results=pd.DataFrame()
    print("Недостаточно валидных наблюдений для квантильного анализа.")


,bucket,count,mean,median,feature
0,"(-1.001, -0.93]",99999,7.297378e-07,0.0,trade_imbalance_20
1,"(-0.93, -0.471]",100003,7.363608e-07,0.0,trade_imbalance_20
2,"(-0.471, 0.233]",99995,-1.710015e-07,0.0,trade_imbalance_20
3,"(0.233, 0.835]",99999,-3.548330e-07,0.0,trade_imbalance_20
4,"(0.835, 1.0]",99999,-9.535263e-07,0.0,trade_imbalance_20
5,"(-1.001, -0.93]",99999,7.297414e-07,0.0,notional_imbalance_20
6,"(-0.93, -0.471]",99999,7.390178e-07,0.0,notional_imbalance_20
7,"(-0.471, 0.233]",99999,-1.727619e-07,0.0,notional_imbalance_20
8,"(0.233, 0.835]",99999,-3.505103e-07,0.0,notional_imbalance_20
9,"(0.835, 1.0]",99999,-9.587131e-07,0.0,notional_imbalance_20


## 8. Критерии вывода и экспорт

**Поддержка гипотезы** — устойчивое упорядочивание будущих изменений mid-price по квантилям imbalance/OFI, достаточная частота наблюдений и эффект, сопоставимый или превышающий spread/издержки. Корреляция сама по себе не доказывает торговую прибыльность.

Проверить:
- не перепутан ли aggressor side;
- нет ли дублирующихся snapshots / одинаковых timestamp;
- одинаково ли покрыты торговые сессии;
- не слишком ли редки обновления стакана относительно сделок;
- не попадает ли в признак snapshot позже времени сделки.

Если `orderbook` хранит только редкие snapshots, мы честно ограничим выводы этой частотой. Не будем называть его event-by-event OFI.

In [9]:

OUT=ROOT/"research"/"tradeflow"/"mark_1"/"results"/"microstructure_research_outputs"
OUT.mkdir(parents=True,exist_ok=True)
tr.head(0).to_csv(OUT/f"{TICKER}_trade_schema.csv",index=False,encoding="utf-8-sig")
schema.to_csv(OUT/f"{TICKER}_raw_schema_report.csv",index=False,encoding="utf-8-sig")
if not joined.empty:
    joined.to_parquet(OUT/f"{TICKER}_trade_flow_features.parquet",index=False)
if not bucket_results.empty:
    bucket_results.to_csv(OUT/f"{TICKER}_imbalance_bucket_results.csv",index=False,encoding="utf-8-sig")
print("Сохранено в:",OUT.resolve())


Сохранено в: C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\microstructure_research_outputs


In [10]:
from pathlib import Path
import pandas as pd

OUT = Path("microstructure_sample")
OUT.mkdir(exist_ok=True)

# 100 последовательных сделок, включая исходные поля AlgoPack
sample = trades.copy()

sample["ts"] = pd.to_datetime(sample["ts"], utc=True)
sample = sample.sort_values(["ts", "TRADENO"])

sample.head(100).to_csv(
    OUT / "GAZP_trades_100.csv",
    index=False,
    encoding="utf-8-sig"
)

# Проверка уникальности и последовательности TRADENO
trade_check = pd.DataFrame([{
    "rows": len(sample),
    "unique_TRADENO": sample["TRADENO"].nunique(),
    "TRADENO_duplicates": sample["TRADENO"].duplicated().sum(),
    "TRADENO_monotonic": sample["TRADENO"].is_monotonic_increasing,
    "unique_timestamps": sample["ts"].nunique(),
    "same_timestamp_rows": int(sample["ts"].duplicated(keep=False).sum()),
}])

trade_check.to_csv(
    OUT / "GAZP_trade_check.csv",
    index=False,
    encoding="utf-8-sig"
)

# Статистика по исходным торговым дням
daily = (
    trades.groupby("__file_date")
    .agg(
        trades=("TRADENO", "size"),
        unique_tradeno=("TRADENO", "nunique"),
        unique_timestamps=("ts", "nunique"),
        volume=("QUANTITY", "sum"),
        turnover=("VALUE", "sum")
    )
    .reset_index()
)

daily.to_csv(
    OUT / "GAZP_daily_stats.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Готово:", OUT.resolve())

Готово: C:\Users\Сергей Кравцов\microstructure_sample
